# Parallel-groups lab

Extract one page's raw vectors and cluster them exactly as P3 **VectorClassification** does: layer -> color ->
stroke-width buckets -> seqno-overlap merge -> constrained spatial clustering.

Then, **within each cluster**, take the **straight** Vectors (all items `'l'`, every point within
`STRAIGHT_TOL_PT` of one line) and group them by **direction**: angle within `ANGLE_TOL_DEG`, folded to
[0, 180), at any offset.

Layers (open with `scripts/pipeline_report_viewer.py`):
- `parallel / angle`: HSV colour with saturation 100 %.
  - **Hue = cluster.**
  - **Value = group angle**, as a triangle wave: 0/180 deg -> 25 %, 90 deg -> 75 %. So 45 deg and 135 deg
    share a value.
- `gradient / vector count` and `gradient / length std`: lowest -> highest, coloured green -> yellow -> red ->
  purple.
- `other / singletons` and `other / excluded (non-straight)`: grey context.
- `clusters / cluster bboxes`: context.

Gradient ranges are page-wide and only cover groups with >= 2 members.

## 0 - Config

In [ ]:
PDF_PATH = None         # None -> first PDF under references/
PAGE = 0
ANGLE_TOL_DEG = 1.0     # single-linkage angle tolerance (folded to [0, 180))
STRAIGHT_TOL_PT = 0.25  # max point distance from the fitted line for a Vector to count as straight
LINE_WIDTH = 1.0

## 1 - Bootstrap

In [ ]:
import colorsys
import os, sys
from pathlib import Path

_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import matplotlib.pyplot as plt
import numpy as np

from rastervec.commons.helpers.geometry import union_bbox
from rastervec.notebooks import _vector_probe_helpers as h

if PDF_PATH is None:
    refs = sorted((Path(_root) / 'references').glob('*.pdf'))
    assert refs, 'no PDFs found under references/'
    PDF_PATH = str(refs[0])
print(PDF_PATH, 'page', PAGE)

## 2 - Extract + cluster (P3 VectorClassification chain)

In [ ]:
page_meta, vectors = h.load_page_vectors(PDF_PATH, PAGE)
clusters = h.cluster_vectors(vectors, page_meta)
print(f'{len(vectors)} vectors -> {len(clusters)} clusters')
report = h.DebugReport('parallel_groups_lab', PDF_PATH, page_meta, line_width=LINE_WIDTH)
hues = h.golden_hues(len(clusters))


def angle_layer(groups, group_cluster, stage):
    """`<stage>` layer: multi-member groups, hue = cluster, value = group angle."""
    colour, vecs = {}, []
    for g, ci in zip(groups, group_cluster):
        if len(g) < 2:
            continue
        rgb = h.hsv_cluster_angle(hues[ci], h._mean_axis_angle([fit.angle for _, fit in g]))
        for v, _ in g:
            colour[id(v)] = rgb
            vecs.append(v)
    report.add_vectors(stage, 'angle (hue=cluster, value=angle)', h.hsv_cluster_angle(0.6, 90),
                       vecs, lambda v: colour[id(v)])


def top_groups(groups, group_cluster, n=15):
    rows = sorted(((g, ci) for g, ci in zip(groups, group_cluster) if len(g) >= 2), key=lambda r: -len(r[0]))[:n]
    print(f"{'cluster':>7} {'n':>5} {'angle':>7} {'mean len':>9} {'len std':>8}")
    for g, ci in rows:
        count, std = h.group_stats(g)
        angle = h._mean_axis_angle([fit.angle for _, fit in g])
        print(f'{ci:>7} {count:>5} {angle:>7.2f} {np.mean([f.length for _, f in g]):>9.2f} {std:>8.3f}')


report.add_boxes('clusters', 'cluster bboxes', '#888888', [
    (union_bbox([v.bbox for v in members]), colorsys.hsv_to_rgb(hues[ci], 1.0, 0.6))
    for ci, (_, members) in enumerate(clusters)
])

## 3 - Parallel straight Vectors within each cluster

In [ ]:
v_groups, v_cluster, v_excluded = [], [], []
for ci, (key, members) in enumerate(clusters):
    straight, rest = h.split_straight(members, STRAIGHT_TOL_PT)
    v_excluded.extend(rest)
    for g in h.group_parallel(straight, ANGLE_TOL_DEG):
        v_groups.append(g)
        v_cluster.append(ci)
v_summary = h.summarise_groups(v_groups)
print(v_summary.describe(len(v_excluded), 'non-straight vector(s)'))

angle_layer(v_groups, v_cluster, 'parallel')
h.add_gradient_layers(report, v_summary, v_excluded, excluded_label='excluded (non-straight)')
fig = h.group_histograms(v_summary, 'Parallel')
report.save_figure(fig, 'histograms')
plt.show()
top_groups(v_groups, v_cluster)

## 4 - Write manifest

In [ ]:
out = report.finish()
print('wrote', out)
for e in report.layers:
    print(f"  {e['stage']:>14} | {e['layer']}")
print()
print('open with:')
print(report.viewer_command())